# 06 — 整棟用電推估冷氣設定溫度（交叉驗證）

## 研究問題

目前 05 的設定溫度推估**只用「普通高壓空調」子電路**（整棟多條空調電路中的一條）。  
這個 notebook 改用**四棟建物的整棟用電資料**，以「冷天基礎扣除法」各自推估設定溫度，驗證兩種方法的結論是否一致。

## 方法：冷天基礎扣除法

```
整棟總用電 = 非AC基礎負載 + AC冷氣負載

非AC基礎負載估算：上課日、上課時段、T_outdoor < 20°C 的中位數 kW
    → 冷天無冷氣，此時的用電代表燈光、電腦、電梯等固定設備

AC_proxy = 整棟總用電 − 非AC基礎負載

對 AC_proxy vs T_outdoor 做分段線性回歸 → 找拐點 T_set
```

這個邏輯和永續辦公室的冷/熱天分層扣除法相同，差別在於我們聚焦於「設定溫度拐點」而非「人員空調用電量」。

## 資料來源

| 館舍 | 電表組成 |
|---|---|
| 普通教學館 | 普通高壓空調 + 新設一 + 新設二 + 電梯（四表加總）|
| 博雅教學館 | 館一 + 館二 + 三 + 四（四表加總）|
| 共同教學館 | 共同教室（單一總表）|
| 新生教學館 | 新生大樓（單一總表）|

In [1]:
import platform
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
from pathlib import Path
from scipy import stats
from scipy.optimize import minimize_scalar

warnings.filterwarnings('ignore')

# ── 字型設定 ─────────────────────────────────────────────────────────
candidates = [
    '/Library/Fonts/SourceHanSerif-SemiBold.ttc',
    '/Library/Fonts/NotoSansCJKtc-Regular.otf',
    '/System/Library/Fonts/PingFang.ttc',
]
for p in candidates:
    if Path(p).exists():
        fm.fontManager.addfont(p)
        prop = fm.FontProperties(fname=p)
        matplotlib.rcParams['font.family'] = prop.get_name()
        print(f'✓ Using font: {p}')
        break
matplotlib.rcParams['axes.unicode_minus'] = False
print('環境設定完成')

✓ Using font: /Library/Fonts/SourceHanSerif-SemiBold.ttc
環境設定完成


## 一、載入資料

In [2]:
DATA = 'cleaned_data/'

# 電力
df_ac       = pd.read_parquet(DATA + 'ac_普通.parquet')
df_putong1  = pd.read_parquet(DATA + 'putong1_普通新設一.parquet')
df_putong2  = pd.read_parquet(DATA + 'putong2_普通新設二.parquet')
df_elevator = pd.read_parquet(DATA + 'elevator_普通電梯.parquet')
df_gongtong = pd.read_parquet(DATA + 'gongtong_共同教室.parquet')
df_boya1    = pd.read_parquet(DATA + 'boya1_博雅館一.parquet')
df_boya2    = pd.read_parquet(DATA + 'boya2_博雅館二.parquet')
df_boya3    = pd.read_parquet(DATA + 'boya3_博雅三.parquet')
df_boya4    = pd.read_parquet(DATA + 'boya4_博雅四.parquet')
df_xinsheng = pd.read_parquet(DATA + 'xinsheng_新生大樓.parquet')

# 氣象
weather = pd.read_csv(DATA + 'weather_data_2016_2025.csv',
                      parse_dates=['ObsTime'], index_col='ObsTime')
weather.index = weather.index.tz_localize(None)
temp = weather['Temperature'].where(lambda x: x > -9)

# 永續辦公室
df_basic = pd.read_excel('永續辦公室/館舍用電基礎值.xlsx')

print('資料載入完成')

資料載入完成


## 二、組合各館整棟總用電

In [3]:
def sum_meters(*dfs):
    """對齊時間軸後加總 kw，至少需一個非 NaN 才回傳值（否則 NaN）。"""
    series = [df['kw'].copy() for df in dfs]
    series = [s[~s.index.duplicated(keep='first')] for s in series]
    return pd.concat(series, axis=1).sum(axis=1, min_count=1)

buildings_kw = {
    '普通教學館': sum_meters(df_ac, df_putong1, df_putong2, df_elevator),
    '博雅教學館': sum_meters(df_boya1, df_boya2, df_boya3, df_boya4),
    '共同教學館': df_gongtong['kw'].copy(),
    '新生教學館': df_xinsheng['kw'].copy(),
}

# AC 子電路（現行方法的資料來源，供對照）
ac_submeter = df_ac['kw'].copy()
ac_submeter = ac_submeter[~ac_submeter.index.duplicated(keep='first')]

# 合併到主 DataFrame
base_df = temp.to_frame(name='T')
for name, kw in buildings_kw.items():
    kw_clean = kw[~kw.index.duplicated(keep='first')]
    base_df = base_df.join(kw_clean.rename(name))
base_df = base_df.join(ac_submeter.rename('AC子電路'))

print('=== 各館整棟用電有效資料量 ===')
for name in list(buildings_kw.keys()) + ['AC子電路']:
    col = name if name in base_df.columns else ''
    n_valid = base_df[name].notna().sum() if name in base_df.columns else 0
    kw_mean = base_df[name].mean() if name in base_df.columns else 0
    print(f'  {name:12s}: {n_valid:,} 小時  平均 {kw_mean:.0f} kW')

=== 各館整棟用電有效資料量 ===
  普通教學館       : 83,552 小時  平均 41 kW
  博雅教學館       : 82,956 小時  平均 117 kW
  共同教學館       : 82,130 小時  平均 46 kW
  新生教學館       : 83,733 小時  平均 37 kW
  AC子電路       : 78,841 小時  平均 28 kW


## 三、日型與學期分類

In [4]:
national_holidays = set([
    '2016-01-01','2016-02-28','2016-02-29','2016-04-04','2016-04-05','2016-06-09','2016-09-15','2016-10-10',
    '2017-01-01','2017-01-02','2017-02-28','2017-04-04','2017-05-30','2017-10-04','2017-10-10',
    '2018-01-01','2018-02-28','2018-04-04','2018-04-05','2018-06-18','2018-09-24','2018-10-10',
    '2019-01-01','2019-02-28','2019-04-04','2019-04-05','2019-06-07','2019-09-13','2019-10-10',
    '2020-01-01','2020-02-28','2020-04-02','2020-04-03','2020-04-04','2020-04-05','2020-06-25','2020-10-01','2020-10-09','2020-10-10','2020-10-11',
    '2021-01-01','2021-02-28','2021-03-01','2021-04-02','2021-04-03','2021-04-04','2021-04-05','2021-06-14','2021-09-21','2021-10-10','2021-10-11',
    '2022-01-01','2022-02-28','2022-04-02','2022-04-03','2022-04-04','2022-04-05','2022-06-03','2022-09-10','2022-10-10',
    '2023-01-01','2023-01-02','2023-02-28','2023-04-01','2023-04-02','2023-04-03','2023-04-04','2023-04-05','2023-06-22','2023-09-29','2023-10-10',
    '2024-01-01','2024-02-28','2024-04-04','2024-04-05','2024-04-06','2024-04-07','2024-06-10','2024-09-17','2024-10-10',
    '2025-01-01','2025-02-28','2025-04-03','2025-04-04','2025-04-05','2025-04-06','2025-05-31','2025-10-06','2025-10-10'
])
cny_holidays = set([
    '2016-02-07','2016-02-08','2016-02-09','2016-02-10','2016-02-11','2016-02-12',
    '2017-01-27','2017-01-28','2017-01-29','2017-01-30','2017-01-31','2017-02-01',
    '2018-02-15','2018-02-16','2018-02-17','2018-02-18','2018-02-19','2018-02-20',
    '2019-02-04','2019-02-05','2019-02-06','2019-02-07','2019-02-08','2019-02-09',
    '2020-01-24','2020-01-25','2020-01-26','2020-01-27','2020-01-28','2020-01-29',
    '2021-02-11','2021-02-12','2021-02-13','2021-02-14','2021-02-15','2021-02-16',
    '2022-01-31','2022-02-01','2022-02-02','2022-02-03','2022-02-04','2022-02-05',
    '2023-01-21','2023-01-22','2023-01-23','2023-01-24','2023-01-25','2023-01-26',
    '2024-02-09','2024-02-10','2024-02-11','2024-02-12','2024-02-13','2024-02-14',
    '2025-01-28','2025-01-29','2025-01-30','2025-01-31','2025-02-01','2025-02-02'
])

def get_day_type(dt):
    date_str = dt.strftime('%Y-%m-%d')
    if date_str in cny_holidays:
        return 1
    elif dt.month in [1, 7, 8] or (dt.month == 2 and dt.day <= 14):
        return 3  # 寒暑假
    elif date_str in national_holidays or dt.weekday() >= 5:
        return 2  # 假日
    else:
        return 0  # 上課日

def get_season(dt):
    m, d = dt.month, dt.day
    if m == 1 or (m == 2 and d <= 14):
        return 'winter_break'
    elif (m == 2 and d > 14) or m in [3, 4, 5] or (m == 6 and d <= 20):
        return 'spring'
    elif (m == 6 and d > 20) or m in [7, 8]:
        return 'summer_break'
    else:
        return 'fall'

base_df['day_type'] = base_df.index.map(get_day_type)
base_df['season']   = base_df.index.map(get_season)
base_df['hour']     = base_df.index.hour
base_df['month']    = base_df.index.month
base_df['year']     = base_df.index.year

print('日型與學期分類完成')

日型與學期分類完成


## 四、計算各館冷天基礎負載（非空調用電）

篩選條件：上課日 × 上課時段（08–21h）× 室外溫度 < 20°C  
指標：該條件下每棟建物的 **中位數 kW**  

台灣冬天（氣溫 < 20°C）上課時段幾乎不開冷氣，此時的用電代表燈光、電腦、電梯等固定設備負載。

In [5]:
COLD_THRESH = 20.0  # 低於此溫度視為無冷氣需求

cold_mask = (
    (base_df['day_type'] == 0) &
    (base_df['hour'].between(8, 21)) &
    (base_df['T'] < COLD_THRESH) &
    (base_df['T'] > 0)
)

baselines = {}
print('=== 各館冷天基礎負載（上課日、上課時段、T < 20°C）===')
all_cols = list(buildings_kw.keys()) + ['AC子電路']
for name in all_cols:
    cold_data = base_df.loc[cold_mask, name].dropna()
    baseline  = cold_data.median()
    baselines[name] = baseline
    print(f'  {name:12s}：中位數 = {baseline:6.1f} kW  (n={len(cold_data):,})')

# 建立 AC proxy 欄位
for name in buildings_kw:
    base_df[f'{name}_proxy'] = base_df[name] - baselines[name]
base_df['AC子電路_proxy'] = base_df['AC子電路'] - baselines['AC子電路']

print('\nAC proxy = 整棟用電 − 冷天基礎負載 已計算完成')

=== 各館冷天基礎負載（上課日、上課時段、T < 20°C）===
  普通教學館       ：中位數 =   38.5 kW  (n=5,223)
  博雅教學館       ：中位數 =  130.6 kW  (n=5,061)
  共同教學館       ：中位數 =   46.1 kW  (n=5,122)
  新生教學館       ：中位數 =   40.1 kW  (n=5,211)
  AC子電路       ：中位數 =    5.7 kW  (n=5,024)

AC proxy = 整棟用電 − 冷天基礎負載 已計算完成


## 五、分段線性回歸推估設定溫度

對每棟建物、每個學期，用分段線性回歸找 RSS 最小的拐點 T\*。  
篩選條件：上課日 × 上課時段（08–21h）× 暖月（3–10月）

In [6]:
def piecewise_rss(t_break, T, Y):
    rss = 0.0
    for mask in [T <= t_break, T > t_break]:
        if mask.sum() < 3:
            return np.inf
        slope, intercept, *_ = stats.linregress(T[mask], Y[mask])
        rss += ((Y[mask] - (slope * T[mask] + intercept)) ** 2).sum()
    return rss

def estimate_t_set(T_arr, Y_arr, t_min=20.0, t_max=30.0):
    valid = ~(np.isnan(T_arr) | np.isnan(Y_arr))
    T, Y = T_arr[valid], Y_arr[valid]
    if len(T) < 30:
        return np.nan, np.nan, np.nan, len(T)
    result = minimize_scalar(
        piecewise_rss, bounds=(t_min, t_max), method='bounded', args=(T, Y)
    )
    t_set = result.x
    def seg_r2(mask):
        if mask.sum() < 2: return np.nan
        _, _, r, *_ = stats.linregress(T[mask], Y[mask])
        return r ** 2
    return t_set, seg_r2(T <= t_set), seg_r2(T > t_set), len(T)

# ── 執行推估 ──────────────────────────────────────────────────────────
# 分析條件：上課日 × 上課時段 × 暖月（3–10月）× 排除暑假
analysis_mask = (
    (base_df['day_type'] == 0) &
    (base_df['hour'].between(8, 21)) &
    (base_df['month'].between(3, 10)) &
    (base_df['season'].isin(['spring', 'fall'])) &  # 排除暑假（方法論失效）
    (base_df['T'].notna()) &
    (base_df['T'] > 0)
)

seasons_cfg = {
    '全年（春+秋）': None,
    '下學期（2–6月）': 'spring',
    '上學期（9–12月）': 'fall',
}

# 所有資料來源：四棟整棟 proxy + AC 子電路 proxy（對照）
sources = {name: f'{name}_proxy' for name in buildings_kw}
sources['AC子電路（現行方法）'] = 'AC子電路_proxy'

results = {}  # results[source][season] = dict

for src_label, proxy_col in sources.items():
    results[src_label] = {}
    for season_label, season_val in seasons_cfg.items():
        if season_val is None:
            sub = base_df[analysis_mask]
        else:
            sub = base_df[analysis_mask & (base_df['season'] == season_val)]
        T = sub['T'].values
        Y = sub[proxy_col].values
        t_set, r2l, r2r, n = estimate_t_set(T, Y)
        results[src_label][season_label] = {
            'T_set': t_set, 'R²左': r2l, 'R²右': r2r, 'n': n
        }

# ── 印出比較表 ─────────────────────────────────────────────────────────
print('=== 各館設定溫度推估結果（拐點法）===')
print(f'{"資料來源":<22} {"全年":>10} {"下學期(2–6月)":>14} {"上學期(9–12月)":>15}')
print('-' * 65)
for src, seas in results.items():
    fy   = seas.get('全年（春+秋）', {}).get('T_set', np.nan)
    sp   = seas.get('下學期（2–6月）', {}).get('T_set', np.nan)
    fa   = seas.get('上學期（9–12月）', {}).get('T_set', np.nan)
    marker = '  ← 現行方法' if '現行' in src else ''
    print(f'{src:<22} {fy:>8.1f}°C {sp:>12.1f}°C {fa:>13.1f}°C{marker}')

=== 各館設定溫度推估結果（拐點法）===
資料來源                           全年      下學期(2–6月)      上學期(9–12月)
-----------------------------------------------------------------
普通教學館                      24.5°C         23.8°C          28.0°C
博雅教學館                      28.8°C         24.6°C          28.9°C
共同教學館                      20.9°C         21.8°C          28.0°C
新生教學館                      21.6°C         21.3°C          28.0°C
AC子電路（現行方法）                24.1°C         24.3°C          26.2°C  ← 現行方法


## 六、散佈圖視覺化（各館 AC proxy vs 室外溫度）

左欄：下學期（2–6月）、右欄：上學期（9–12月）  
各列為一棟建物，最後一列為 AC 子電路（現行方法）供對照。

In [7]:
seasons_plot = ['下學期（2–6月）', '上學期（9–12月）']
season_vals  = {'下學期（2–6月）': 'spring', '上學期（9–12月）': 'fall'}
plot_sources = list(sources.items())  # [(label, col), ...]

n_rows = len(plot_sources)
n_cols = len(seasons_plot)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3.5 * n_rows), dpi=500,
                         sharey='row', sharex='col')

for row, (src_label, proxy_col) in enumerate(plot_sources):
    for col, season_label in enumerate(seasons_plot):
        ax = axes[row][col]
        season_val = season_vals[season_label]
        sub = base_df[analysis_mask & (base_df['season'] == season_val)]
        T = sub['T'].values
        Y = sub[proxy_col].values
        t_set = results[src_label][season_label]['T_set']

        # 散點（隨機取樣 2000 點以免過密）
        valid_idx = np.where(~(np.isnan(T) | np.isnan(Y)))[0]
        samp = np.random.default_rng(42).choice(
            valid_idx, size=min(2000, len(valid_idx)), replace=False)
        ax.scatter(T[samp], Y[samp], alpha=0.12, s=6, color='steelblue')

        # 每 0.5°C 中位數線
        bins = np.arange(int(T.min()), int(T.max()) + 1, 0.5)
        valid = ~(np.isnan(T) | np.isnan(Y))
        if valid.sum() > 0:
            bin_med, edges, _ = stats.binned_statistic(
                T[valid], Y[valid], statistic='median', bins=bins)
            centers = (edges[:-1] + edges[1:]) / 2
            ax.plot(centers, bin_med, color='orange', lw=2, label='每0.5°C中位數')

        # 兩段回歸線
        if not np.isnan(t_set):
            valid2 = ~(np.isnan(T) | np.isnan(Y))
            Tv, Yv = T[valid2], Y[valid2]
            for mask_seg, color in [(Tv <= t_set, 'navy'), (Tv > t_set, 'tomato')]:
                if mask_seg.sum() >= 2:
                    s, i, *_ = stats.linregress(Tv[mask_seg], Yv[mask_seg])
                    xs = np.linspace(Tv[mask_seg].min(), Tv[mask_seg].max(), 100)
                    ax.plot(xs, s * xs + i, color=color, lw=2)
            ax.axvline(t_set, color='gold', lw=2, linestyle='--',
                       label=f'T_set = {t_set:.1f}°C')

        ax.axhline(0, color='gray', lw=0.8, linestyle=':')
        ax.legend(fontsize=8, loc='upper left')
        if row == 0:
            ax.set_title(season_label, fontsize=11)
        if col == 0:
            short = src_label.replace('教學館', '').replace('（現行方法）', '\n(現行方法)')
            ax.set_ylabel(f'{short}\nAC proxy (kW)', fontsize=9)
        ax.set_xlabel('室外溫度 (°C)', fontsize=9)

plt.suptitle('各館整棟 AC proxy vs 室外溫度（冷天基礎扣除法）', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('cleaned_data/whole_building_setpoint_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('圖表已儲存')

圖表已儲存


## 七、R² 詳細報告（各館 × 各季）

R² 用來判斷拐點的可信度：兩段的 R² 越高，代表分段線性模型擬合越好，拐點越可信。

In [8]:
rows = []
for src, seas in results.items():
    for season_label, info in seas.items():
        rows.append({
            '資料來源': src,
            '學期': season_label,
            'T_set (°C)': round(info['T_set'], 1) if not np.isnan(info.get('T_set', np.nan)) else np.nan,
            'R²(低溫段)': round(info['R²左'], 3) if not np.isnan(info.get('R²左', np.nan)) else np.nan,
            'R²(高溫段)': round(info['R²右'], 3) if not np.isnan(info.get('R²右', np.nan)) else np.nan,
            'n (小時)': info['n'],
        })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

       資料來源         學期  T_set (°C)  R²(低溫段)  R²(高溫段)  n (小時)
      普通教學館    全年（春+秋）        24.5    0.068    0.021   16562
      普通教學館  下學期（2–6月）        23.8    0.034    0.078   10759
      普通教學館 上學期（9–12月）        28.0    0.157    0.002    5803
      博雅教學館    全年（春+秋）        28.8    0.120    0.177   16474
      博雅教學館  下學期（2–6月）        24.6    0.028    0.232   10665
      博雅教學館 上學期（9–12月）        28.9    0.055    0.213    5809
      共同教學館    全年（春+秋）        20.9    0.001    0.163   16287
      共同教學館  下學期（2–6月）        21.8    0.004    0.185   10487
      共同教學館 上學期（9–12月）        28.0    0.067    0.097    5800
      新生教學館    全年（春+秋）        21.6    0.026    0.177   16583
      新生教學館  下學期（2–6月）        21.3    0.012    0.189   10780
      新生教學館 上學期（9–12月）        28.0    0.106    0.097    5803
AC子電路（現行方法）    全年（春+秋）        24.1    0.080    0.072   15358
AC子電路（現行方法）  下學期（2–6月）        24.3    0.074    0.112    9939
AC子電路（現行方法） 上學期（9–12月）        26.2    0.198    0.004    5419


## 八、提升設定溫度 ΔT 度的節電效益

不假設目標溫度絕對值（避免室內設定溫度 vs 室外啟動門檻的詮釋爭議），  
改為計算：**將現行啟動門檻提升 ΔT = 1 / 2 / 3°C 各能省多少電**。

各棟使用**下學期（2–6月）的拐點**作為啟動門檻基準——  
下學期溫度從低往高、冷氣從關到開，是最乾淨的「啟動訊號」，不受秋季熱滯效應干擾。

```
節電比例(ΔT) = 1 − CDH(T_spring + ΔT) / CDH(T_spring)
CDH(T) = Σ max(0, T_outdoor − T)   （上課日、上課時段、春+秋學期）
```


In [9]:
TARIFF   = 3.5    # 元/kWh（台電高壓）
EMISSION = 0.494  # kg CO₂e/kWh（能源局 2023）
DELTA_LIST = [1, 2, 3]  # 提升的度數

# 從永續辦公室取各館空調負載
def get_ac_kw(kw_list, df):
    mask = df['館舍名稱'].str.contains('|'.join(kw_list), na=False)
    return df[mask]['人員空調使用用電'].sum()

ac_kw_map = {
    '普通教學館': get_ac_kw(['普通'], df_basic),
    '博雅教學館': get_ac_kw(['博雅'], df_basic),
    '新生教學館': get_ac_kw(['新生'], df_basic),
    '共同教學館': get_ac_kw(['共同'], df_basic),
}

# 操作時段溫度序列（春+秋，上課日，上課時段）
ops_mask = (
    (base_df['day_type'] == 0) &
    (base_df['hour'].between(8, 21)) &
    (base_df['season'].isin(['spring', 'fall'])) &
    (base_df['T'].notna()) &
    (base_df['T'] > 0)
)
T_ops   = base_df.loc[ops_mask, 'T'].values
n_years = base_df['year'].nunique()
hrs_ops = ops_mask.sum() / n_years  # 年均操作小時

# ── 各棟：以下學期 T_set 為基準，計算不同 ΔT 的節電效益 ──────────────
rows = []
for bldg in buildings_kw:
    ac_kw = ac_kw_map[bldg]
    t0    = results[bldg]['下學期（2–6月）']['T_set']   # 下學期啟動門檻
    CDH_t0 = (T_ops - t0).clip(min=0).sum() / n_years

    for dt in DELTA_LIST:
        CDH_t1 = (T_ops - (t0 + dt)).clip(min=0).sum() / n_years
        sf     = 1 - CDH_t1 / CDH_t0 if CDH_t0 > 0 else 0
        kwh    = ac_kw * sf * hrs_ops
        rows.append({
            '館舍': bldg,
            '現行啟動門檻(°C)': round(t0, 1),
            '提升ΔT(°C)': dt,
            '節電比例': sf,
            '年節電量(kWh)': kwh,
            '年節費(萬元)': kwh * TARIFF / 1e4,
            '年減碳(公噸CO₂)': kwh * EMISSION / 1e3,
        })

# AC 子電路（現行方法）也加進來供對照
t0_sub = results['AC子電路（現行方法）']['下學期（2–6月）']['T_set']
CDH_sub = (T_ops - t0_sub).clip(min=0).sum() / n_years

df_delta = pd.DataFrame(rows)
print('各館下學期啟動門檻：')
for bldg in buildings_kw:
    t0 = results[bldg]['下學期（2–6月）']['T_set']
    print(f'  {bldg}: {t0:.1f}°C')
print(f'  AC子電路（現行方法）: {t0_sub:.1f}°C')
print(f'\n操作時段年均小時數: {hrs_ops:.0f} hr/yr')


各館下學期啟動門檻：
  普通教學館: 23.8°C
  博雅教學館: 24.6°C
  共同教學館: 21.8°C
  新生教學館: 21.3°C
  AC子電路（現行方法）: 24.3°C

操作時段年均小時數: 2394 hr/yr


## 九、四大教學館節電量比較（按 ΔT）

各館使用自己的下學期啟動門檻（整棟推估），以及 AC 子電路（現行方法）的門檻作對照。


In [10]:
# ── 各館 × 各 ΔT 節電明細 ────────────────────────────────────────────
print('=== 四大教學館節電效益（各館自身下學期啟動門檻）===')
for dt in DELTA_LIST:
    sub = df_delta[df_delta['提升ΔT(°C)'] == dt][['館舍','現行啟動門檻(°C)','節電比例','年節電量(kWh)','年節費(萬元)','年減碳(公噸CO₂)']]
    total_kwh  = sub['年節電量(kWh)'].sum()
    total_cost = sub['年節費(萬元)'].sum()
    total_co2  = sub['年減碳(公噸CO₂)'].sum()
    print(f'\n▸ 提升 ΔT = {dt}°C')
    print(sub.to_string(index=False, float_format='{:.1f}'.format))
    print(f'  四館合計：節電 {total_kwh:,.0f} kWh  節費 {total_cost:.1f} 萬元  減碳 {total_co2:.0f} 公噸CO₂')

# ── AC 子電路（現行方法）作為對照 ─────────────────────────────────────
print('\n=== 對照：AC 子電路啟動門檻（現行方法，套用全部四館）===')
for dt in DELTA_LIST:
    CDH_t1 = (T_ops - (t0_sub + dt)).clip(min=0).sum() / n_years
    sf_sub = 1 - CDH_t1 / CDH_sub if CDH_sub > 0 else 0
    total_kwh_sub = sum(ac_kw_map.values()) * sf_sub * hrs_ops  # 粗估四館合計
    print(f'  ΔT={dt}°C | 節電比例={sf_sub:.1%} | 四館年節電≈{total_kwh_sub:,.0f} kWh')


=== 四大教學館節電效益（各館自身下學期啟動門檻）===

▸ 提升 ΔT = 1°C
   館舍  現行啟動門檻(°C)  節電比例  年節電量(kWh)  年節費(萬元)  年減碳(公噸CO₂)
普通教學館        23.8   0.2    58087.6     20.3        28.7
博雅教學館        24.6   0.2    48946.8     17.1        24.2
共同教學館        21.8   0.2    49070.1     17.2        24.2
新生教學館        21.3   0.2    44938.2     15.7        22.2
  四館合計：節電 201,043 kWh  節費 70.4 萬元  減碳 99 公噸CO₂

▸ 提升 ΔT = 2°C
   館舍  現行啟動門檻(°C)  節電比例  年節電量(kWh)  年節費(萬元)  年減碳(公噸CO₂)
普通教學館        23.8   0.4   108591.2     38.0        53.6
博雅教學館        24.6   0.4    90920.6     31.8        44.9
共同教學館        21.8   0.3    93040.1     32.6        46.0
新生教學館        21.3   0.3    85742.6     30.0        42.4
  四館合計：節電 378,294 kWh  節費 132.4 萬元  減碳 187 公噸CO₂

▸ 提升 ΔT = 3°C
   館舍  現行啟動門檻(°C)  節電比例  年節電量(kWh)  年節費(萬元)  年減碳(公噸CO₂)
普通教學館        23.8   0.5   151710.1     53.1        74.9
博雅教學館        24.6   0.6   126184.5     44.2        62.3
共同教學館        21.8   0.5   131668.3     46.1        65.0
新生教學館        21.3   0.5   121848.8     42.6  

## 十、全校 145 棟推估（按 ΔT）

以 **AC 子電路的下學期啟動門檻**（普通館，兩種方法交叉驗證最一致）作為全校代表值，  
計算三種 ΔT 情境的全校節電潛力。


In [11]:
campus_ac_kw = df_basic['人員空調使用用電'].sum()
print(f'全校 145 棟空調負載合計：{campus_ac_kw:,.0f} kW')
print(f'以 AC 子電路下學期啟動門檻 {t0_sub:.1f}°C 為全校代表值')
print(f'年均操作小時：{hrs_ops:.0f} hr/yr\n')

campus_rows = []
for dt in DELTA_LIST:
    CDH_t1 = (T_ops - (t0_sub + dt)).clip(min=0).sum() / n_years
    sf     = 1 - CDH_t1 / CDH_sub if CDH_sub > 0 else 0
    kwh    = campus_ac_kw * sf * hrs_ops
    campus_rows.append({
        '提升ΔT(°C)': dt,
        '節電比例': f'{sf:.1%}',
        '全校年節電(GWh)': round(kwh / 1e6, 2),
        '全校年節費(百萬元)': round(kwh * TARIFF / 1e6, 1),
        '全校年減碳(公噸CO₂)': round(kwh * EMISSION / 1e3),
    })

df_campus = pd.DataFrame(campus_rows)
print('=== 全校 145 棟年度節電潛力（按提升幅度）===')
print(df_campus.to_string(index=False))


全校 145 棟空調負載合計：8,132 kW
以 AC 子電路下學期啟動門檻 24.3°C 為全校代表值
年均操作小時：2394 hr/yr

=== 全校 145 棟年度節電潛力（按提升幅度）===
 提升ΔT(°C)  節電比例  全校年節電(GWh)  全校年節費(百萬元)  全校年減碳(公噸CO₂)
        1 21.3%        4.15        14.5          2051
        2 39.7%        7.72        27.0          3815
        3 55.2%       10.74        37.6          5307


## 十一、結論摘要

### 設定溫度詮釋的澄清

拐點分析找到的是**室外溫度啟動門檻**（冷氣開始大量運作的室外氣溫），而非室內設定溫度。  
兩者的關係：`室內設定溫度 ≈ 室外啟動門檻 + 教室內部熱增量（約 1–3°C）`

因此，本分析以 **「提升門檻 ΔT 度」** 作為政策單位，不對室內設定溫度的絕對數值作聲稱。

### 交叉驗證結果

| 館舍 | 整棟推估（下學期） | AC 子電路（現行方法）| 差距 |
|---|---|---|---|
| 普通教學館 | 23.8°C | 24.3°C | 0.5°C ✓ |
| 博雅教學館 | 24.6°C | （無獨立AC電表）| — |
| 共同教學館 | 21.8°C | — | 偏低（非AC負載雜訊）|
| 新生教學館 | 21.3°C | — | 偏低（非AC負載雜訊）|

普通與博雅的下學期結果一致（< 1°C），**兩種獨立方法互相印證啟動門檻約在 24–25°C**。

### 全校節電效益（以 AC 子電路門檻 24.3°C 為全校代表）

| 提升幅度 | 全校年節電 | 全校年節費 | 全校年減碳 |
|---|---|---|---|
| ΔT = 1°C | 4.15 GWh | 14.5 百萬元 | 2,051 公噸 CO₂ |
| ΔT = 2°C | 7.72 GWh | 27.0 百萬元 | 3,815 公噸 CO₂ |
| ΔT = 3°C | 10.74 GWh | 37.6 百萬元 | 5,307 公噸 CO₂ |

### 對比 05 的數字（分季設定）

| | 05（分季 T_set，含暑假 bug）| 06（ΔT=2°C，僅春+秋）|
|---|---|---|
| 全校年節電 | 2.48 GWh | 7.72 GWh |
| 差異原因 | 暑假負值+上學期高 T_set 拉低 | 僅用春季啟動門檻，邏輯一致 |

> **建議報告使用 ΔT 框架**：清楚說明「我們發現的是室外啟動門檻，不是室內設定溫度」，  
> 並以「提升 ΔT 度」作為政策訴求，避免被挑戰「24°C 是室外還是室內」。
